# Synthetic Text Diversity Evaluation

This notebook evaluates lexical diversity of synthetic data using metrics from Texygen and related benchmarks:

1. **Distinct-n** – Fraction of unique n-grams in the corpus. Higher values indicate more diverse text; low values suggest mode collapse or repetition.
2. **Self-BLEU** – BLEU of each synthetic sentence against the rest of the corpus. Lower values indicate more diverse generations; high Self-BLEU warns of repetitive or low-entropy text.

## 1. Setup

In [9]:
%pip install -q pandas nltk


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
from typing import List, Tuple
import nltk

nltk.download("punkt_tab", quiet=True)

True

## 2. Diversity Metrics

In [11]:
def tokenize(text: str) -> List[str]:
    """Tokenize text (whitespace split). Swap for language-specific tokenizer if needed."""
    return str(text).strip().split() if pd.notna(text) and str(text).strip() else []


def get_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    """Extract n-grams from token list."""
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)] if len(tokens) >= n else []


def compute_distinct_n(texts: List[str], n: int = 2) -> dict:
    """
    Compute Distinct-n: fraction of unique n-grams in the corpus.
    Higher = more diverse. Low values suggest mode collapse or repetition.

    distinct_n = |unique n-grams| / |total n-grams|
    """
    all_ngrams = []
    for text in texts:
        tokens = tokenize(text)
        all_ngrams.extend(get_ngrams(tokens, n))

    total = len(all_ngrams)
    unique = len(set(all_ngrams))
    distinct = unique / total if total > 0 else 0.0

    return {"unique_ngrams": unique, "total_ngrams": total, f"distinct_{n}": distinct}

In [12]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def compute_self_bleu(texts: List[str], max_n: int = 4, sample_size: int = None) -> dict:
    """
    Compute Self-BLEU: BLEU of each sentence against the rest of the corpus.
    Lower = more diverse. High Self-BLEU warns of repetitive or low-entropy text.

    Uses geometric mean of BLEU-1 to BLEU-4 (Texygen-style).
    """
    tokenized = [tokenize(t) for t in texts if tokenize(t)]
    if len(tokenized) < 2:
        return {"self_bleu": 0.0, "n_sentences": len(tokenized)}

    if sample_size and len(tokenized) > sample_size:
        import random
        tokenized = random.sample(tokenized, sample_size)

    smoothing = SmoothingFunction().method1
    scores = []

    for i, hyp in enumerate(tokenized):
        refs = [tok for j, tok in enumerate(tokenized) if j != i]
        if not refs or not hyp:
            continue
        bleu_n = []
        for n in range(1, max_n + 1):
            if len(hyp) >= n:
                w = [1.0 / n] * n
                s = sentence_bleu(refs, hyp, weights=tuple(w), smoothing_function=smoothing)
                bleu_n.append(s)
        if bleu_n:
            geo_mean = 1.0
            for s in bleu_n:
                geo_mean *= s
            geo_mean **= 1.0 / len(bleu_n)
            scores.append(geo_mean)

    avg = sum(scores) / len(scores) if scores else 0.0
    return {"self_bleu": avg, "n_sentences": len(scores)}

## 3. Load Data

In [13]:
import os

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "data", "processed")
if not os.path.exists(DATA_DIR):
    DATA_DIR = "data/processed"
synthetic_path = os.path.join(DATA_DIR, "train_1500_para_final.csv")
original_path = os.path.join(DATA_DIR, "train_processed.csv")

synthetic_df = pd.read_csv(synthetic_path)
text_column = "Sentence_clean" if "Sentence_clean" in synthetic_df.columns else "Sentence"
synthetic_texts = synthetic_df[text_column].dropna().astype(str).tolist()
print(f"Synthetic dataset: {len(synthetic_texts)} samples")

Synthetic dataset: 10541 samples


## 4. Compute Diversity Metrics

In [14]:
# Distinct-n (n=1,2,3)
print("Computing Distinct-n...")
distinct_1 = compute_distinct_n(synthetic_texts, n=1)
distinct_2 = compute_distinct_n(synthetic_texts, n=2)
distinct_3 = compute_distinct_n(synthetic_texts, n=3)

print("\n=== Distinct-n ===")
print(f"Distinct-1: {distinct_1['distinct_1']:.4f} (unique unigrams: {distinct_1['unique_ngrams']:,} / {distinct_1['total_ngrams']:,})")
print(f"Distinct-2: {distinct_2['distinct_2']:.4f} (unique bigrams: {distinct_2['unique_ngrams']:,} / {distinct_2['total_ngrams']:,})")
print(f"Distinct-3: {distinct_3['distinct_3']:.4f} (unique trigrams: {distinct_3['unique_ngrams']:,} / {distinct_3['total_ngrams']:,})")

Computing Distinct-n...

=== Distinct-n ===
Distinct-1: 0.0393 (unique unigrams: 6,211 / 158,046)
Distinct-2: 0.4389 (unique bigrams: 64,746 / 147,505)
Distinct-3: 0.8032 (unique trigrams: 110,018 / 136,968)


In [15]:
# Self-BLEU (use sample_size=500 for faster run on large corpora)
print("Computing Self-BLEU...")
self_bleu_result = compute_self_bleu(synthetic_texts, sample_size=500)
print(f"\n=== Self-BLEU ===")
print(f"Self-BLEU: {self_bleu_result['self_bleu']:.4f} (n_sentences: {self_bleu_result['n_sentences']})")
print("\nInterpretation: Lower Self-BLEU = more diverse. High Self-BLEU (>0.5) suggests repetitive text.")

Computing Self-BLEU...

=== Self-BLEU ===
Self-BLEU: 0.2871 (n_sentences: 500)

Interpretation: Lower Self-BLEU = more diverse. High Self-BLEU (>0.5) suggests repetitive text.


## 5. Optional: Compare with Original Data

In [16]:
if os.path.exists(original_path):
    original_df = pd.read_csv(original_path)
    orig_text_col = "Sentence_clean" if "Sentence_clean" in original_df.columns else "Sentence"
    original_texts = original_df[orig_text_col].dropna().astype(str).tolist()
    print(f"Original dataset: {len(original_texts)} samples")

    orig_d1 = compute_distinct_n(original_texts, n=1)
    orig_d2 = compute_distinct_n(original_texts, n=2)
    orig_sb = compute_self_bleu(original_texts, sample_size=500)

    summary = pd.DataFrame({
        "Metric": ["Distinct-1", "Distinct-2", "Distinct-3", "Self-BLEU"],
        "Original": [orig_d1["distinct_1"], orig_d2["distinct_2"], compute_distinct_n(original_texts, 3)["distinct_3"], orig_sb["self_bleu"]],
        "Synthetic": [distinct_1["distinct_1"], distinct_2["distinct_2"], distinct_3["distinct_3"], self_bleu_result["self_bleu"]],
    })
    summary["Synthetic - Original"] = summary["Synthetic"] - summary["Original"]
    display(summary)
else:
    print("Original file not found. Skipping comparison.")

Original dataset: 5548 samples


,Metric,Original,Synthetic,Synthetic - Original
0,Distinct-1,0.059729,0.039299,-0.020430
1,Distinct-2,0.593515,0.438941,-0.154574
2,Distinct-3,0.930389,0.803239,-0.127150
3,Self-BLEU,0.252482,0.287109,0.034627
